In [ ]:
# --- Бібліотеки для даної роботи ---
try:
    import markdown, numpy, pandas, plotly, nbformat
    print("Бібліотеки вже встановлені. Пропускаємо інсталяцію.")
except ImportError:
    print("Встановлюємо бібліотеки...")
    %pip install markdown numpy pandas plotly nbformat

<div align="center">
    <h2><b>Architecture Design Document (ADD)</b></h2>
    <h1>Проектування глобальної E-Commerce платформи (Multi-Region)</h1>
    <h3>Глобальний маркетплейс «2BeMarket»</h3>
    <br>

**Метадані документа:**

| **Тип документа:** | High-Level Design (HLD) & Global Scale |
| :--- | :--- |
| **Статус:** | Draft / В розробці |
| **Масштаб:** | Global (Multi-Region Active-Active) |
| **Версія архітектури:** | 1.0.0 |
| **Дата створення:** | Березень 2026 р. |

<br>

**Автор проекту:**

| **Ім'я:** | Олег Гаценко |
| :--- | :--- |
| **Роль:** | Principal System Architect / Студент магістратури |
| **Організація:** | NeoVersity (Master's in AI & Machine Learning) |

</div>

---

### **Executive Summary (Резюме архітектури):**

Цей документ описує комплексне проектування високорівневої архітектури (HLD) для глобальної платформи електронної комерції **2BeMarket** (аналог Amazon / AliExpress). Платформа розрахована на понад **100 млн DAU** та роботу в умовах екстремальних навантажень (Global Black Friday). Головний архітектурний виклик — забезпечення мілісекундної затримки для покупців по всьому світу зі збереженням суворої консистентності фінансових транзакцій.

**Ключові архітектурні парадигми та рішення:**
- **Multi-Region Active-Active:** Розгортання інфраструктури у трьох незалежних географічних зонах (США, Європа, Азія) за допомогою Global Server Load Balancing (GSLB). Це гарантує виживання системи навіть у разі падіння цілого континенту (Disaster Recovery).
- **Conflict-Free Carts (CRDT):** Кошики користувачів реалізовані на базі безконфліктних структур даних (CRDT) у розподіленому NoSQL-сховищі (DynamoDB/Cassandra), що виключає втрату товарів при перемиканні користувача між регіонами.
- **Distributed Transactions & Saga:** Відмова від повільних 2PC-транзакцій на користь патерну **Saga** (Оркестрація) з використанням `Idempotency Keys`. Це гарантує надійність списання коштів та інвентаризації складських залишків.
- **Global Polyglot Persistence:** Використання NewSQL (наприклад, Google Spanner / CockroachDB) для транзакційного ядра з суворою консистентністю (ACID) на глобальному рівні, та ElasticSearch для миттєвого повнотекстового пошуку товарів.
- **Machine Learning & Personalization:** Асинхронний пайплайн (Kafka + Flink) для збору клікстріму та віддачі персоналізованих рекомендацій у режимі реального часу як для веб-клієнтів, так і для **мобільних застосунків**.
- **Multi-Cloud & Data Sovereignty:** Інфраструктура є Cloud-Agnostic (Kubernetes) і розгорнута в різних хмарах (AWS для Америки, GCP для Європи/Швейцарії, Alibaba для Азії). Це гарантує юридичну відповідність суворим місцевим законам про зберігання персональних даних (GDPR, FADP, PIPL).
- **Edge Security & Geo-Fencing:** Використання Cloudflare WAF на рівні глобального маршрутизатора для захисту від DDoS та жорсткого геоблокування (Geo-Ban) підсанкційних регіонів (наприклад, рф) ще до потрапляння трафіку у внутрішню мережу маркетплейсу.

### **Контекстна діаграма потоків (Level 0):**

```mermaid
graph LR
    %% Styling (Luxury & Dark Theme)
    classDef source fill:#2b2b2b,stroke:#00ffcc,stroke-width:2px,color:#fff
    classDef stream fill:#1a1a1a,stroke:#ff3366,stroke-width:2px,color:#fff
    classDef batch fill:#1a1a1a,stroke:#3399ff,stroke-width:2px,color:#fff
    classDef sink fill:#2b2b2b,stroke:#ffcc00,stroke-width:2px,color:#fff
    classDef banned fill:#3a0000,stroke:#ff0000,stroke-width:2px,color:#fff
    
    %% Нові Люксові Стилі
    classDef buyer_lux fill:#5e35b1,stroke:#ffd700,stroke-width:3px,color:#fff,rx:10px,ry:10px
    classDef seller_lux fill:#00796b,stroke:#ffd700,stroke-width:3px,color:#fff,rx:10px,ry:10px

    Buyer(("📱 Покупці<br>(Мобільний застосунок / Web)")):::buyer_lux
    Seller(("💻 Продавці<br>(Merchant Portal)")):::seller_lux
    Sanctioned(("🚫 Підсанкційний трафік<br>(рф тощо)")):::banned

    WAF{{"🛡️ Edge Security & WAF<br>(Cloudflare / Geo-Ban)"}}:::source
    GLB{{"🌐 Global Load Balancer<br>& API Gateway"}}:::source

    Catalog["🛒 Catalog & Search<br>(ElasticSearch + Redis)"]:::batch
    Checkout["💳 Cart & Checkout<br>(Saga Orchestrator)"]:::stream

    Bus[["⚡ Global Event Bus<br>(Apache Kafka)"]]:::source

    Storage[("🗄️ Polyglot Persistence<br>(NewSQL, Cassandra)")]:::sink
    Bank[["🏦 Платіжні Шлюзи<br>(Stripe/PayPal)"]]:::sink
    Logistics[["📦 Логістичні Хаби<br>(DHL/FedEx/Local Post)"]]:::sink

    %% Зв'язки безпеки та роутингу
    Buyer -- "Мільйони запитів/сек<br>(Пошук, Кошик)" --> WAF
    Seller -- "Управління інвентарем" --> WAF
    Sanctioned -. "Connection Dropped" .-> WAF
    WAF -- "Авторизований трафік" --> GLB

    %% Внутрішні потоки
    GLB -- "Read-heavy трафік" --> Catalog
    GLB -- "Write-heavy (Idempotent)" --> Checkout

    Checkout -- "Синхронна оплата" --> Bank
    Checkout -- "Асинхронні події" --> Bus

    Bus -- "Eventual Consistency" --> Storage
    Checkout -. "ACID Транзакції" .-> Storage
    Catalog -. "Індексація даних" .-> Storage
    Bus -- "Push: Інтеграція доставки" --> Logistics
```

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **1. Системні вимоги та Back-of-the-envelope аналіз**

Проект **2BeMarket** створюється як глобальна екосистема, що поєднує покупців та продавців у різних юрисдикціях. Тому вимоги до системи охоплюють не лише технічні, але й суворі юридичні та логістичні аспекти.

### 1.1. Функціональні вимоги (Functional Requirements)

1. **Каталог та Пошук:** Користувачі (**мобільний застосунок** / Web) можуть шукати товари, фільтрувати їх за десятками параметрів та переглядати деталі в режимі реального часу.
2. **Кошик та Оформлення (Checkout):** Можливість додавати товари в кошик, змінювати їх кількість та безшовно переходити до оплати через зовнішні платіжні шлюзи (Stripe, PayPal, Alipay).
3. **Управління замовленнями та Логістика:** Відстеження статусу замовлення, система повернень, а також інтеграція з логістичними хабами (Logistics Gateway) для оновлення трекінг-статусів.
4. **Управління користувачами та Продавцями:** Реєстрація покупців та B2B-портал для продавців (Merchant Portal) з управлінням інвентарем та зонами доставки.
5. **Персоналізація:** Генерація рекомендацій ("Часто купують разом") на основі історії покупок та клікстріму.
6. **Аналітика:** Надання агрегованої статистики продажів для продавців та інвесторів.

---

### 1.2. Нефункціональні вимоги (Non-Functional Requirements)

1. **Висока доступність (High Availability):** Цільова доступність 99.999% (не більше 5 хвилин простою на рік). Збереження працездатності навіть при падінні цілого AWS-регіону.
2. **Низька затримка (Low Latency):** Час відповіді API < 200 мс для 99% запитів; повнотекстовий пошук серед мільйонів товарів < 500 мс у будь-якій точці планети.
3. **Строга консистентність (Strong Consistency):** Гарантія ACID-транзакцій для фінансових операцій та списання залишків зі складу (уникнення проблеми подвійного продажу одного товару).
4. **Data Sovereignty & Комплаєнс:** Суворе дотримання законів про локалізацію персональних даних (GDPR в ЄС, FADP у Швейцарії, PIPL у Китаї). Дані користувачів не повинні перетинати кордони своїх регіонів без дозволу.
5. **Інтернаціоналізація (i18n):** Динамічна підтримка кількох мов, конвертація валют у реальному часі та автоматичний розрахунок регіональних податків (Tax Engine).

---

### 1.3. Оцінка навантаження та ресурсів (Capacity Planning)

Для доведення масштабованості системи наведемо базові (back-of-the-envelope) розрахунки для цільової аудиторії:

- **Трафік:** 100 млн активних користувачів щодня (DAU). При середньому показнику 15 запитів на користувача маємо **1.5 млрд запитів/день**, що дорівнює базовому навантаженню у **~17,500 RPS** (Requests Per Second).
- **Піковий RPS (Black Friday / Флеш-розпродажі):** Історично трафік у такі дні зростає в 10 разів. Система повинна бути спроектована на витримування **175,000 RPS**. З них приблизно 5% — це транзакції запису (Write), тобто **~8,750 TPS** (Transactions Per Second), що вимагає потужного брокера повідомлень (Kafka) та оптимізованого пулу підключень до БД.
- **Пропускна здатність (Bandwidth):** Середній розмір відповіді API (JSON-метадані + посилання на CDN) складає близько 50 KB. Базовий вихідний трафік: 17,500 RPS × 50 KB ≈ **875 MB/sec** (близько 7 Gbps на рівні балансувальників).
- **Зберігання даних (Storage):** Очікується близько 50 млн нових замовлень на день. Середній розмір об'єкта замовлення (JSON) ≈ 2 KB. 
    - Приріст транзакційної БД: 50 млн × 2 KB = **100 GB/day**.
    - За 5 років (з урахуванням 3-кратної реплікації для надійності): 100 GB × 365 × 5 × 3 = **~165 TB** високошвидкісного дискового простору лише для ядра транзакцій. Це математично обґрунтовує відмову від монолітного PostgreSQL на користь георозподілених NewSQL рішень (CockroachDB / Spanner).

**Візуалізація профілю навантаження під час пікових подій:**
```mermaid
pie title Розподіл трафіку (Black Friday - 175,000 RPS)
    "Перегляд каталогу та Пошук (Read-heavy)" : 85
    "Генерація рекомендацій (ML Inference)" : 10
    "Кошик та Оформлення (Write-heavy / ACID)" : 5
```

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **2. Високорівнева архітектура (High-Level Design)**

Головний архітектурний виклик **2BeMarket** — це не просто масштабування до 175,000 RPS, а забезпечення суверенітету даних (Data Sovereignty) та уникнення Vendor Lock-in у глобальному масштабі. Для цього система відмовляється від монохмарного підходу на користь гібридної мультихмарної екосистеми.

### 2.1. Глобальна мультихмарна топологія (Multi-Cloud Geo-Strategy)

Щоб мінімізувати затримки (latency), забезпечити юридичну відповідність місцевим законам та уникнути геополітичних ризиків, інфраструктура розгорнута на чотирьох різних платформах:

- **Amazon Web Services (AWS):** Основний бекбон для Північної та Південної Америки, а також для **Східної Азії (Японія, Південна Корея, Тайвань)**. Розгортання в азійських дата-центрах AWS (Tokyo, Seoul) дозволяє уникнути геополітичних ризиків використання китайських провайдерів для цих юрисдикцій, водночас забезпечуючи максимальну ємність.
- **Google Cloud Platform (GCP):** Використовується для Європи (включаючи Швейцарію). GCP обрано через сувору відповідність європейським стандартам безпеки (GDPR, FADP), а також передові інструменти для дата-аналітики.
- **Microsoft Azure:** Покриває регіони з особливими вимогами до корпоративної інфраструктури: **Африка, Австралія, Океанія та Близький Схід (Middle East)**. Використання Azure в ОАЕ та Катарі дозволяє ізолювати транзакційне ядро для інтеграції зі специфічними шлюзами ісламського банкінгу (Sharia Compliant Payments).
- **Alibaba Cloud:** Використовується виключно для **материкового Китаю та країн Південно-Східної Азії** (Індонезія, Малайзія) для дотримання місцевого законодавства (PIPL) та інтеграції з локальними платіжними системами (Alipay, WeChat Pay). Міжхмарний зв'язок (Cross-Cloud) із зовнішнім світом забезпечується через виділені канали Cloud Enterprise Network (CEN).

---

### 2.2. Архітектурна схема глобального роутингу та Federated Analytics

```mermaid
graph TD
    %% Styling (Dark Theme & Luxury)
    classDef source fill:#2b2b2b,stroke:#00ffcc,stroke-width:2px,color:#fff
    classDef mesh fill:#1a1a1a,stroke:#ffcc00,stroke-width:2px,color:#fff
    classDef banned fill:#3a0000,stroke:#ff0000,stroke-width:2px,color:#fff
    classDef dark_circle fill:#0a0a0a,stroke:#ff0000,stroke-width:3px,color:#fff
    classDef lux fill:#5e35b1,stroke:#ffd700,stroke-width:3px,color:#fff,rx:10px,ry:10px

    Client(("📱 Клієнти<br>(Global Traffic)")):::lux
    Sanctioned(("🚫 Підсанкційний трафік<br>(рф тощо)")):::banned

    CF{{"🛡️ Cloudflare<br>Global CDN & WAF (Geo-IP)"}}:::source

    %% Принциповий Geo-Ban
    Sanctioned -.->|Geo-IP Block| CF
    CF -.->|Drop Connection| Blocked(("❌ 403 Forbidden")):::dark_circle

    Client -->|HTTPS / GraphQL| CF

    %% Глобальний шар аналітики (Global Data Mesh)
    subgraph Global_Mesh ["Глобальна Мережа Даних / Global Data Mesh (Cloud-Agnostic)"]
		
        Kafka{"⚡ Apache Kafka<br>(Крос-хмарний MirrorMaker)"}:::mesh
        GlobalLake[("🗄️ Озеро Даних / Global Data Lake<br>(Знеособлені Дані)")]:::mesh
        InvestorDash["📊 Дашборди<br>для Інвесторів"]:::mesh
        
        Kafka --> GlobalLake --> InvestorDash
    end

    %% --- ФУНДАМЕНТАЛЬНИЙ РІВЕНЬ ХМАРНИХ ПРОВАЙДЕРІВ ---

    subgraph AWS ["🟧&nbsp;Amazon&nbsp;Web&nbsp;Services&nbsp;(AWS)<br>Америка та Сх. Азія (Японія/Корея/Тайвань)"]
        GW_US["API Gateway"]
        Order_US["Мікросервіси (K8s)"]
        DB_US[("Локальна БД / Local DB")]
        Agg_US["Агрегатор (Spark)"]
        
        GW_US --> Order_US --> DB_US --> Agg_US
    end

    subgraph GCP ["🟩&nbsp;Google&nbsp;Cloud&nbsp;Platform&nbsp;(GCP)<br>Європа та Швейцарія"]
        GW_EU["API Gateway"]
        Order_EU["Мікросервіси (K8s)"]
        DB_EU[("Локальна БД<br>(GDPR/FADP Compliant)")]
        Agg_EU["Агрегатор (Spark)"]
        
        GW_EU --> Order_EU --> DB_EU --> Agg_EU
    end

    subgraph AZURE ["🟦 Microsoft Azure<br>Африка, Австралія, Океанія та Бл. Схід"]
        GW_MEA["API Gateway"]
        Order_MEA["Мікросервіси (K8s)"]
        DB_MEA[("Локальна БД<br>(Sharia/Local Laws)")]
        Agg_MEA["Агрегатор (Spark)"]
        
        GW_MEA --> Order_MEA --> DB_MEA --> Agg_MEA
    end

    subgraph ALI ["🟥 Alibaba Cloud<br>Материковий Китай та Півд.-Сх. Азія"]
        GW_AS["API Gateway"]
        Order_AS["Мікросервіси (K8s)"]
        DB_AS[("Локальна БД<br>(PIPL Compliant)")]
        Agg_AS["Агрегатор (Spark)"]
        
        GW_AS --> Order_AS --> DB_AS --> Agg_AS
    end

    %% Маршрутизація трафіку від CDN до хмар
    CF -->|Geo-Routing| GW_US
    CF -->|Geo-Routing| GW_EU
    CF -->|Geo-Routing| GW_MEA
    CF -->|Geo-Routing| GW_AS

    %% Передача знеособлених даних у глобальну шину
    Agg_US -->|Знеособлені метрики| Kafka
    Agg_EU -->|Знеособлені метрики| Kafka
    Agg_MEA -->|Знеособлені метрики| Kafka
    Agg_AS -->|Знеособлені метрики| Kafka
```

---

### 2.3. Опис ключових глобальних компонентів

1. **Edge Security & Geo-Routing (Cloudflare):** Точка входу в систему. Використовує Anycast-маршрутизацію для направлення клієнта до найближчої хмари. Тут же на рівні CDN реалізовано жорсткий Geo-Ban: трафік з несанкціонованих юрисдикцій просто відкидається (Drop Connection), захищаючи бекенд від "сміттєвого" навантаження.
2. **Local Regions (Cloud-Agnostic):** Оскільки ми одночасно працюємо в AWS, GCP, Azure та Alibaba, мікросервіси упаковані в **Kubernetes (K8s)**. Це дозволяє команді інженерів використовувати єдиний CI/CD пайплайн (наприклад, ArgoCD) незалежно від того, в якій хмарі розгортається код. Персональні дані клієнтів (PII) зберігаються виключно в **Local DB** відповідного регіону тобто в межах баз даних свого регіону і ніколи не перетинають кордони (Compliance by Design).
3. **Federated Analytics & Global Data Mesh:** Для надання єдиної статистики інвесторам/продавцям та формування звітності використовується федеративна аналітика. Локальні агрегатори (Apache Spark) обробляють сирі дані всередині кожної хмари, повністю видаляють імена та адреси, і генерують агреговані бізнес-метрики (наприклад, "продано 10,000 смартфонів у Цюриху"). Лише ці **знеособлені метрики** пересилаються через крос-хмарну шину (Kafka MirrorMaker) у Global Data Lake, що унеможливлює порушення законів про суверенітет даних.

<hr style="height: 15px; border: none; background: linear-gradient(to right, #007bff, #6610f2, #e83e8c, #fd7e14, #ffc107, #28a745); border-radius: 2px;">

## **3. Дизайн даних (Polyglot Persistence) та API**

У глобальній системі з навантаженням до 175,000 RPS використання єдиної монолітної реляційної бази даних є архітектурним антипатерном (Single Point of Failure та вузьке місце масштабування). Тому **2BeMarket** використовує підхід **Polyglot Persistence** — застосування різних спеціалізованих сховищ даних для різних бізнес-доменів.

### 3.1. Вибір баз даних та стратегії зберігання

Кожен регіональний кластер (AWS, GCP тощо) містить власну ізольовану комбінацію сховищ:

1.  **Транзакційне ядро (Orders & Payments): NewSQL (CockroachDB / Google Spanner)**
    - *Чому:* Нам потрібні суворі ACID-транзакції для фінансів, але з можливістю горизонтального масштабування (на відміну від класичного PostgreSQL).
    - *Стратегія:* Синхронна реплікація в межах регіону (Multi-AZ) для забезпечення високої доступності (High Availability).
2.  **Каталог та Пошук (Product Catalog): ElasticSearch + Redis**
    - *Чому:* 85% трафіку — це пошук товарів. ElasticSearch забезпечує миттєвий повнотекстовий та фасетний пошук, а кластер Redis діє як Distributed Cache для найпопулярніших товарів (стратегія *Cache-Aside*).
3.  **Кошики користувачів (Shopping Carts): NoSQL Wide-Column (Apache Cassandra)**
    - *Чому:* Кошики генерують гігантську кількість операцій запису (Write-heavy). Cassandra ідеально масштабується на запис. 
    - *Фіча:* Використання безконфліктних структур даних (CRDT) для об'єднання кошиків, якщо користувач змінив пристрій або регіон під час покупок.
4.  **Сховище медіа (Images & Videos): Object Storage (S3 / GCS)**
    - *Чому:* Зберігання терабайтів фотографій товарів. Доступ до них здійснюється виключно через глобальну CDN (Cloudflare).

---

### 3.2. Схема даних (ER Diagram)

Нижче наведено концептуальну модель даних (Entity-Relationship) для ключових сутностей транзакційного ядра.

```mermaid
erDiagram
    %% Entities
    USER {
        uuid user_id PK
        string email
        string password_hash
        string region "EU, US, AS, MEA"
        datetime created_at
    }
    
    PRODUCT {
        uuid product_id PK
        string seller_id FK
        string name
        decimal price
        int stock_quantity
        string status "ACTIVE, OUT_OF_STOCK, BANNED"
    }
    
    CART {
        uuid cart_id PK
        uuid user_id FK
        datetime updated_at
        string status "OPEN, CHECKED_OUT"
    }
    
    CART_ITEM {
        uuid cart_id FK
        uuid product_id FK
        int quantity
    }
    
    ORDER {
        uuid order_id PK
        uuid user_id FK
        decimal total_amount
        string status "PENDING, PAID, SHIPPED, DELIVERED"
        string idempotency_key "UK (Унікальний ключ для безпеки)"
        datetime created_at
    }
    
    ORDER_ITEM {
        uuid order_id FK
        uuid product_id FK
        int quantity
        decimal locked_price "Ціна на момент покупки"
    }

    %% Relationships
    USER ||--o{ ORDER : "places"
    USER ||--|| CART : "owns"
	PRODUCT ||--o{ CART_ITEM : "added as"
    CART ||--o{ CART_ITEM : "contains"
    ORDER ||--|{ ORDER_ITEM : "includes"
    PRODUCT ||--o{ ORDER_ITEM : "ordered as"
```

---

### 3.3. Дизайн API (Ключові контракти)

Для взаємодії між мобільним застосунком / веб-клієнтом та API Gateway використовується RESTful підхід (з елементами GraphQL для складних запитів каталогу). Нижче наведено 4 критично важливі ендпоінти платформи.

1. **Пошук товарів (Read-Heavy API):**
    - **Ендпоінт:** `GET /api/v1/products/search`
    - **Джерело даних:** ElasticSearch + Redis Cache
    - **Параметри запиту:** `?q=laptop&category=electronics&sort=price_asc&limit=20&page=1`
    - **Відповідь (200 OK):** Повертає масив об'єктів товарів. Затримка < 100мс.
2. **Додавання товару в кошик (Write-Heavy API):**
    - **Ендпоінт:** `POST /api/v1/cart/items`
    - **Джерело даних:** Apache Cassandra
    - **Тіло запиту (Payload):**
		```json
		{
		"product_id": "uuid-1234",
		"quantity": 1
		}
		```
	- **Особливість:** Операція є ідемпотентною (патерн *Upsert*). Якщо клієнт відправить той самий запит двічі через збій мережі, кількість товарів не подвоїться помилково, а перезапишеться до актуального стану.
3. **Оформлення замовлення та Оплата (Transactional API):**
    - **Ендпоінт:** `POST /api/v1/orders/checkout`
    - **Джерело даних:** CockroachDB (Запускає Saga Pattern)
    - **Заголовки (Headers):** `Idempotency-Key: <unique_uuid>` (Критично важливо для запобігання подвійному списанню коштів).
    - **Тіло запиту:**
		```json
		{
		"cart_id": "uuid-5678",
		"payment_method_id": "pm_9876",
		"shipping_address_id": "addr_123"
		}
		```
	- **Відповідь (202 Accepted):** Оскільки транзакція розподілена, API не чекає завершення платежу. Воно повертає `order_id` зі статусом `PENDING`, а подальша обробка йде асинхронно.
4. **Перевірка статусу замовлення (Async Polling):**
    - **Ендпоінт:** `GET /api/v1/orders/{order_id}/status`
    - **Опис:** Використовується мобільним застосунком для періодичного опитування (Polling) або через WebSocket (Server-Sent Events) для отримання оновлень статусу після виклику Checkout (наприклад, перехід від `PENDING` до `PAID`).

---

### 3.4. Обробка граничних випадків та Anti-Fraud логіка (Edge Cases)

У глобальному середовищі поведінка користувачів є непередбачуваною. Архітектура **2BeMarket** закладає стійкість до наступних сценаріїв:

1.  **Cross-Device Synchronization (Web vs. Mobile):** Кошики користувачів прив'язані до `user_id` у розподіленій базі Cassandra. Використання патерну CRDT (Conflict-free Replicated Data Types) гарантує, що якщо користувач додасть товар з мобільного застосунку, а потім продовжить покупки через Web-інтерфейс, стани кошиків безконфліктно об'єднаються без втрати даних.
2.  **Cross-Region Hop (Використання VPN):** Якщо користувач змінює гео-локацію (наприклад, вмикає VPN США) перед самим оформленням замовлення, Global Load Balancer перенаправить його трафік у новий хмарний регіон (наприклад, з GCP до AWS). Глобальна реплікація баз даних гарантує наявність кошика в новому регіоні. Проте перед ініціалізацією оплати система примусово викликає `Tax Engine` для перерахунку доступності та цін.
3.  **Розділення Billing та Shipping Address (Сценарій "Подарунок"):** Система не робить жорсткої прив'язки методів оплати до адреси доставки. Це дозволяє здійснювати транскордонні покупки (наприклад, оплата швейцарською карткою для доставки в іншу країну).
    - *Compliance:* Митні правила та податки розраховуються суворо за **Shipping Address** (адресою доставки).
    - *Anti-Fraud:* Розбіжність між Geo-IP, Billing Address та Shipping Address не призводить до автоматичного блокування. Замість цього транзакція отримує підвищений Risk Score, що тригерить додаткову верифікацію клієнта через **3D Secure** на боці платіжного шлюзу (Stripe / PayPal).